In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()))

import pandas as pd
import numpy as np
from config import SILVER_DATASET, GOLD_DASHBOARD, GOLD_DATA_DIR


In [2]:
df = pd.read_csv(SILVER_DATASET)
print(f"Silver rows: {len(df):,}  columns: {df.shape[1]}")
df.head(2)

Silver rows: 24,974  columns: 39


,source_system,ocr_engine,doc_type,document_image_quality,ocr_confidence,ocr_error_count,text_language,normalized_amount,normalized_date,normalization_method,...,income_declared,tenure_months,collateral_type,ltv,pd_model_score,final_decision,requires_human_review,default_12m,is_duplicate,env_risk_level
0,OCR_PDF,AZURE_OCR,EXTRATO,0.821,0.562,4,PT,10348.78,2024-06-19,RULES_V1,...,53421.80,210,IMOVEL,0.67,0.297,APPROVE,1,0,0,BAIXO
1,EMAIL_ATTACH,GOOGLE_VISION,EXTRATO,0.622,0.830,1,PT,21423.14,2025-09-17,RULES_V2,...,28138.01,152,VEICULO,0.67,0.223,APPROVE,0,1,0,BAIXO


## Fraud features

In [3]:
df["fraude_idioma_estrangeiro"] = (df["text_language"] != "PT").astype(int)
df["fraude_sem_garantia"]       = (df["collateral_type"] == "SEM_GARANTIA").astype(int)
df["fraude_match_baixo"]        = (df["match_score"] < 0.4).astype(int)
df["fraude_pii_idioma"]         = ((df["text_language"] != "PT") & (df["pii_detected"] == 0)).astype(int)
df["fraude_muitas_violacoes"]   = (df["rule_violations"] > 2).astype(int)
df["fraude_duplicado"]          = (df["is_duplicate"] == 1).astype(int)
df["fraude_ocr_baixo"]          = (df["ocr_confidence"] < 0.5).astype(int)
df["fraude_compliance_review"]  = (df["compliance_status"] == "REVIEW").astype(int)

sinais = [
    "fraude_idioma_estrangeiro", "fraude_sem_garantia", "fraude_match_baixo",
    "fraude_pii_idioma", "fraude_muitas_violacoes", "fraude_duplicado",
    "fraude_ocr_baixo", "fraude_compliance_review",
]
df["fraude_score"] = df[sinais].sum(axis=1)
df["fraude_risco"] = pd.cut(
    df["fraude_score"],
    bins=[-1, 1, 3, 8],
    labels=["BAIXO", "MEDIO", "ALTO"],
)

## Environmental features

In [4]:
df["drought_bin"] = pd.cut(
    df["drought_spi"],
    bins=[-5, -1.5, -0.5, 0.5, 5],
    labels=["Seca severa", "Seca moderada", "Normal", "Úmido"],
)

## Credit / income derived features

In [5]:
df["ratio_credit_income"] = (
    df["credit_requested_value"] / df["income_declared"].replace(0, np.nan)
).replace([np.inf, -np.inf], np.nan)

df["risk_tier"] = (
    (df["ltv"] > 1.0).astype(int)
    + (df["pd_model_score"] > df["pd_model_score"].quantile(0.75)).astype(int)
    + (df["drought_spi"] < df["drought_spi"].quantile(0.25)).astype(int)
    + (df["flood_risk_idx"] > df["flood_risk_idx"].quantile(0.75)).astype(int)
    + (df["ratio_credit_income"] > 0.6).astype(int)
)

df["ratio"] = (
    df["credit_requested_value"] / df["income_declared"]
).replace([np.inf, -np.inf], np.nan)

df["ltv_faixa"] = pd.cut(
    df["ltv"],
    bins=[0, 0.5, 1.0, 1.5, 2.0, 100],
    labels=["0–50%", "50–100%", "100–150%", "150–200%", ">200%"],
)

## Save gold layer

In [6]:
GOLD_DATA_DIR.mkdir(parents=True, exist_ok=True)
df.to_csv(GOLD_DASHBOARD, index=False)
print(f"Saved → {GOLD_DASHBOARD}")
print(f"Rows: {len(df):,}  columns: {df.shape[1]}")
print("New columns:", [c for c in df.columns if c not in pd.read_csv(SILVER_DATASET, nrows=0).columns])

Saved → /home/paolot/Workspace/BB_data_analysis/data/gold/gold_dashboard.csv
Rows: 24,974  columns: 54


New columns: ['fraude_idioma_estrangeiro', 'fraude_sem_garantia', 'fraude_match_baixo', 'fraude_pii_idioma', 'fraude_muitas_violacoes', 'fraude_duplicado', 'fraude_ocr_baixo', 'fraude_compliance_review', 'fraude_score', 'fraude_risco', 'drought_bin', 'ratio_credit_income', 'risk_tier', 'ratio', 'ltv_faixa']
